In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os

In [ ]:
#set display and plotting options
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [ ]:
# Common path setup for local + Colab

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # CHANGE this path if your repo is in a different folder in Drive
    REPO_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/masters-project")
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT.name != "masters-project" and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent

    if REPO_ROOT.name != "masters-project":
        raise RuntimeError("Could not find repo root folder named 'masters-project'.")

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
REPORTS_EVAL = REPO_ROOT / "reports" / "eval" / "modserve"
FIG_DIR = REPORTS_EVAL / "figures"
TABLE_DIR = REPORTS_EVAL / "tables"
CACHE_DIR = REPORTS_EVAL / "cache"
LOG_DIR = REPORTS_EVAL / "logs"

for p in [DATA_RAW, DATA_PROCESSED, FIG_DIR, TABLE_DIR, CACHE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_TRACE_PATH = DATA_RAW / "AzureLMMInferenceTrace_multimodal.csv.gz"
CLEAN_REQUESTS_PATH = DATA_PROCESSED / "cleaned_requests.parquet"
DAILY_SUMMARY_PATH = TABLE_DIR / "daily_summary.csv"

print("Repo root:", REPO_ROOT)
print("Running in Colab:", IN_COLAB)
print("Raw trace path:", RAW_TRACE_PATH)

In [ ]:
# load the raw dataset
if not RAW_TRACE_PATH.exists():
    raise FileNotFoundError(
        f"Missing raw dataset: {RAW_TRACE_PATH}\n"
        "Place AzureLMMInferenceTrace_multimodal.csv.gz in data/raw/ first."
    )
df = pd.read_csv(RAW_TRACE_PATH)
df.head()

In [ ]:
#inspect columns and shape
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
#standardize column names
df = df.rename(columns={
    "TIMESTAMP": "TIMESTAMP",
    "NumImages": "NumImages",
    "ContextTokens": "ContextTokens",
    "GeneratedTokens": "GeneratedTokens"
})

required_cols = ["TIMESTAMP", "NumImages", "ContextTokens", "GeneratedTokens"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()
df.head()

In [ ]:
#parse timestamps and clean numeric fields
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], errors="coerce", utc=True)

df["NumImages"] = pd.to_numeric(df["NumImages"], errors="coerce")
df["ContextTokens"] = pd.to_numeric(df["ContextTokens"], errors="coerce")
df["GeneratedTokens"] = pd.to_numeric(df["GeneratedTokens"], errors="coerce")

df = df.dropna(subset=["TIMESTAMP", "NumImages", "ContextTokens", "GeneratedTokens"]).copy()

df["NumImages"] = df["NumImages"].clip(lower=0)
df["ContextTokens"] = df["ContextTokens"].clip(lower=0)
df["GeneratedTokens"] = df["GeneratedTokens"].clip(lower=0)

df = df.sort_values("TIMESTAMP").reset_index(drop=True)
df.head()

In [ ]:
#create useful request-level flags
df["is_image_text"] = df["NumImages"] > 0
df["is_text_only"] = df["NumImages"] == 0

df["request_type"] = np.where(df["is_image_text"], "image_text", "text_only")

df["minute"] = df["TIMESTAMP"].dt.floor("min")
df["day"] = df["TIMESTAMP"].dt.floor("D")

df.head()

In [ ]:
#basic sanity checks
print("Number of rows:", len(df))
print("Time range:", df["TIMESTAMP"].min(), "to", df["TIMESTAMP"].max())
print("\nRequest type counts:")
print(df["request_type"].value_counts())

print("\nSummary statistics:")
display(df[["NumImages", "ContextTokens", "GeneratedTokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

In [ ]:
#requests per minute by request type
requests_per_min = (
    df.groupby(["minute", "request_type"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

requests_per_min.plot()
plt.title("Requests per Minute by Request Type")
plt.xlabel("Time")
plt.ylabel("Requests / Minute")
plt.tight_layout()
plt.savefig(FIG_DIR / "requests_per_minute_by_type.png", dpi=180)
plt.show()

In [ ]:
#context tokens per minute by request type
context_tokens_per_min = (
    df.groupby(["minute", "request_type"])["ContextTokens"]
      .sum()
      .unstack(fill_value=0)
      .sort_index()
)

context_tokens_per_min.plot()
plt.title("Context Tokens per Minute by Request Type")
plt.xlabel("Time")
plt.ylabel("Context Tokens / Minute")
plt.tight_layout()
plt.savefig(FIG_DIR / "context_tokens_per_minute_by_type.png", dpi=180)
plt.show()

In [ ]:
#generated tokens per minute by request type
generated_tokens_per_min = (
    df.groupby(["minute", "request_type"])["GeneratedTokens"]
      .sum()
      .unstack(fill_value=0)
      .sort_index()
)

generated_tokens_per_min.plot()
plt.title("Generated Tokens per Minute by Request Type")
plt.xlabel("Time")
plt.ylabel("Generated Tokens / Minute")
plt.tight_layout()
plt.savefig(FIG_DIR / "generated_tokens_per_minute_by_type.png", dpi=180)
plt.show()

In [ ]:
#images per minute
images_per_min = (
    df.groupby("minute")["NumImages"]
      .sum()
      .sort_index()
)

images_per_min.plot()
plt.title("Images per Minute")
plt.xlabel("Time")
plt.ylabel("Images / Minute")
plt.tight_layout()
plt.savefig(FIG_DIR / "images_per_minute.png", dpi=180)
plt.show()

In [ ]:
#helper function for CDF plots
def empirical_cdf(values):
    values = np.asarray(values)
    values = values[~np.isnan(values)]
    values = np.sort(values)
    y = np.arange(1, len(values) + 1) / len(values)
    return values, y

In [ ]:
#CDF of prompt length
plt.figure(figsize=(8, 5))

for label, subset in [("image_text", df[df["is_image_text"]]), ("text_only", df[df["is_text_only"]])]:
    x, y = empirical_cdf(subset["ContextTokens"].values)
    plt.plot(x, y, label=label)

plt.title("CDF of Prompt Length")
plt.xlabel("Context Tokens")
plt.ylabel("CDF")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "cdf_prompt_length.png", dpi=180)
plt.show()

In [ ]:
#CDF of images per request
image_requests = df[df["NumImages"] > 0].copy()

plt.figure(figsize=(8, 5))
x, y = empirical_cdf(image_requests["NumImages"].values)
plt.plot(x, y, label="image_text_requests")

plt.title("CDF of Images per Request")
plt.xlabel("Number of Images")
plt.ylabel("CDF")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "cdf_images_per_request.png", dpi=180)
plt.show()

In [ ]:
#daily summary table
daily_summary = (
    df.groupby("day")
      .agg(
          requests=("TIMESTAMP", "size"),
          image_text_requests=("is_image_text", "sum"),
          text_only_requests=("is_text_only", "sum"),
          total_context_tokens=("ContextTokens", "sum"),
          total_generated_tokens=("GeneratedTokens", "sum"),
          total_images=("NumImages", "sum"),
      )
      .reset_index()
)

daily_summary["frac_image_text"] = daily_summary["image_text_requests"] / daily_summary["requests"]

daily_summary

In [ ]:
# save the daily summary
daily_summary.to_csv(DAILY_SUMMARY_PATH, index=False)
print("Saved:", DAILY_SUMMARY_PATH)

In [ ]:
# save the cleaned request-level dataset
df.to_parquet(CLEAN_REQUESTS_PATH, index=False)
print("Saved cleaned request-level dataset to:", CLEAN_REQUESTS_PATH)